# Decision Tree Regression Experiment

## Objective

To evaluate decision tree regression model to ddetermine whether it can capture complex relationships and improve predictive performance compared to previous linear regression models. 

## Hypothesis

The decision tree model is expected to outperform prior linear regression models because it can capture non linear relationships without manual feature engineering though categorical encoding is still required.

In [1]:
cd ..

c:\Users\user\Desktop\regression-problem


In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [3]:
from src.preprocessing import ModelPreprocessors
mp = ModelPreprocessors()
field_df = mp.load_data()

2026-09-11 10:19:58 | src.data_ingestion | INFO | Starting data ingestion
2026-09-11 10:19:58 | src.preprocessing | INFO | Model preprocessor is initialized
2026-09-11 10:19:58 | src.field_data_processor | INFO | FieldDataProcessor is initialized
2026-09-11 10:19:58 | src.data_ingestion | INFO | Successfully connected to sqlite:///data/Maji_Ndogo_farm_survey_small.db
2026-09-11 10:19:58 | src.data_ingestion | INFO | Query executed successfully. Rows: 5654
2026-09-11 10:19:58 | src.field_data_processor | INFO | SQL data is successfully loaded into the pandas DataFrame
2026-09-11 10:19:58 | src.field_data_processor | INFO | Swapped columns: Annual_yield with Crop_type
2026-09-11 10:19:58 | src.field_data_processor | INFO | Converted the negative elavtion values to absulte figures.
2026-09-11 10:19:58 | src.field_data_processor | INFO | Mispelled crop names and extra spaces were found and got fixed
2026-09-11 10:19:58 | src.data_ingestion | INFO | Attempting to read CSV from: https://raw.

In [4]:
field_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5654 entries, 0 to 5653
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Elevation          5654 non-null   float64
 1   Latitude           5654 non-null   float64
 2   Longitude          5654 non-null   float64
 3   Location           5654 non-null   str    
 4   Slope              5654 non-null   float64
 5   Rainfall           5654 non-null   float64
 6   Min_temperature_C  5654 non-null   float64
 7   Max_temperature_C  5654 non-null   float64
 8   Ave_temps          5654 non-null   float64
 9   Soil_fertility     5654 non-null   float64
 10  Soil_type          5654 non-null   str    
 11  pH                 5654 non-null   float64
 12  Pollution_level    5654 non-null   float64
 13  Plot_size          5654 non-null   float64
 14  Annual_yield       5654 non-null   float64
 15  Crop_type          5654 non-null   str    
 16  Standard_yield     5654 non-null   

In [5]:
X, y = mp.X_y_features()
display(y)

0       0.577964
1       0.486302
2       0.649647
3       0.532348
4       0.555076
          ...   
5649    0.554482
5650    0.438194
5651    0.800776
5652    0.507595
5653    0.453064
Name: Standard_yield, Length: 5654, dtype: float64

In [6]:
X_train, X_test, y_train, y_test = mp.X_y_train_test_split()

2026-09-11 13:23:51 | src.preprocessing | INFO | Train test split is successfully


In [7]:
num_features, cat_features = mp.get_feature_types()
display(num_features)
display(cat_features)

['Elevation',
 'Latitude',
 'Longitude',
 'Slope',
 'Rainfall',
 'Min_temperature_C',
 'Max_temperature_C',
 'Ave_temps',
 'Soil_fertility',
 'pH',
 'Pollution_level',
 'Plot_size']

['Location', 'Soil_type', 'Crop_type']

## Baseline model

In [8]:
def decision_tree_baseline() -> Pipeline:
    transformer = mp.feature_transformer()
    tree_pipeline = Pipeline(
        steps=[
            ("preprocessor",transformer),
            ("model", DecisionTreeRegressor(random_state=42))
        ]
    )
    return tree_pipeline


In [8]:
def tree_baseline_model(X_train, X_test, y_train)-> DecisionTreeRegressor | np.ndarray:
    tree_pipeline = decision_tree_baseline()
    tree_model = tree_pipeline.fit(X_train, y_train)
    y_tree_pred = tree_model.predict(X_test)
    return tree_model, y_tree_pred

tree_model, y_tree_pred = tree_baseline_model(X_train, X_test, y_train)
display(tree_model)

NameError: name 'decision_tree_baseline' is not defined

## Model evaluation

In [ ]:
def tree_evaluation(y_test, y_tree_pred) ->tuple[float, float]:
    tree_rmse = np.sqrt(mean_squared_error(y_test, y_tree_pred))
    tree_r2_score = r2_score(y_test, y_tree_pred)
    return tree_rmse, tree_r2_score

tree_rmse, tree_r2_score = tree_evaluation(y_test, y_tree_pred)
print(f"rmse: {tree_rmse:.4f}")
print(f"r2 score: {tree_r2_score:.4f}")

rmse: 0.0323
r2 score: 0.9218


In [ ]:
def checking_overfitting(X_train, X_test) ->tuple[np.ndarray, np.ndarray]:
    y_train_pred = tree_model.predict(X_train)
    y_test_pred = tree_model.predict(X_test)
    train_rmse = root_mean_squared_error(y_train, y_train_pred)
    test_rmse = root_mean_squared_error(y_test, y_test_pred)
    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)
    return train_rmse, test_rmse, train_r2_score, test_r2_score
train_rmse, test_rmse, train_r2_score, test_r2_score= checking_overfitting(X_train, X_test)
print(f"train rmse: {train_rmse:.4f}")
print(f"tets rmse: {test_rmse:.4f}")

print(f"train r2 score: {train_r2_score:.4f}")
print(f"tets r2 score: {test_r2_score:.4f}")

train rmse: 0.0000
tets rmse: 0.0323
train r2 score: 1.0000
tets r2 score: 0.9218


**The tree has essentially memorized training data and is losing some performance on unseen data**

## inspecting the tree complexity

In [ ]:
tree = tree_model.named_steps["model"]
print("Tree depth:", tree.get_depth())
print("Number of leaves:", tree.get_n_leaves())

Tree depth: 29
Number of leaves: 4523


In [ ]:
def evaluate_decision_tree(tree_params, X_train, X_test,y_train,y_test) -> pd.DataFrame:

    results = []
    for params in tree_params:
        transformer = mp.feature_transformer()
        tree_pipeline = Pipeline(
            steps=[
                ("preprocessor", transformer),
                ("model", DecisionTreeRegressor(random_state=42, **params)
                )
            ]
        )

        tree_pipeline.fit(X_train, y_train)
        y_train_pred = tree_pipeline.predict(X_train)
        y_test_pred = tree_pipeline.predict(X_test)

        results.append({
            **params,
            "train_rmse": np.sqrt(mean_squared_error(y_train, y_train_pred)),
            "test_rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "train_r2": r2_score(y_train, y_train_pred),
            "test_r2": r2_score(y_test, y_test_pred)
        })

    return pd.DataFrame(results)

In [ ]:
np.random.seed(42)
depths_list = np.random.randint(1, 50, size = 50)
leaf_sizes = np.random.randint(1, 50, size = 50)
tree_params = [
    {"max_depth": depth, "min_samples_leaf": leaf_size}
    for depth, leaf_size in zip(depths_list, leaf_sizes)]


In [ ]:
tree_evaluation_results = evaluate_decision_tree( tree_params,X_train, X_test, y_train, y_test)
tree_params = tree_evaluation_results.sort_values("test_r2", ascending=False)
tree_params.head(10)

,max_depth,min_samples_leaf,train_rmse,test_rmse,train_r2,test_r2
13,40,2,0.007552,0.031357,0.995356,0.926213
45,25,1,0.000226,0.031535,0.999996,0.925370
14,24,6,0.018533,0.032099,0.972031,0.922678
12,36,4,0.014358,0.032178,0.983213,0.922295
16,22,4,0.014364,0.032371,0.983199,0.921362
3,43,7,0.020018,0.032565,0.967372,0.920417
28,25,8,0.021040,0.032831,0.963953,0.919113
5,21,8,0.021042,0.032986,0.963947,0.918345
48,26,9,0.022538,0.034135,0.958640,0.912556
29,49,14,0.027971,0.037208,0.936296,0.896104


## Insight

The Decision Tree outperformed the previous models, with the best random configuration (`max_depth=40`, `min_samples_leaf=2`) achieving **RMSE = 0.03136** and **R² = 0.92621**. The strong performance of deeper trees suggests nonlinear relationships in the data, although the train–test gap indicates some overfitting. **Cross-validation will be used next to assess whether this performance generalizes consistently.**


## Cross Validation

**Does the Decision Tree's 0.926 R2 hold consistently, or did train/test split make it look better?**

In [ ]:
from sklearn.model_selection import cross_validate

np.random.seed(42)
depths_list = np.random.randint(1, 50, size=50)
leaf_sizes = np.random.randint(1, 50, size=50)

tree_params = [
    {"model__max_depth": depth, "model__min_samples_leaf": leaf_size}
    for depth, leaf_size in zip(depths_list, leaf_sizes)
]


def tree_cv_pipeline() -> Pipeline:
    transformer = mp.feature_transformer()
    cv_model_pipeline = Pipeline(
        steps=[
            ("preprocessing", transformer),
            ("model", DecisionTreeRegressor(random_state=42)),
        ]
    )
    return cv_model_pipeline


In [ ]:
def tree_cross_validation(tree_params, X_train, y_train):
    results = []
    for params in tree_params:
        cv_model_pipeline = tree_cv_pipeline()
        cv_model_pipeline.set_params(**params)

        cv_results = cross_validate(
            cv_model_pipeline,
            X_train,
            y_train,
            cv=5,
            scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"},
            return_train_score=True,
            n_jobs=-1,
        )

        results.append(
            {
                **params,
                "mean_train_rmse": -cv_results["train_rmse"].mean(),
                "mean_val_rmse": -cv_results["test_rmse"].mean(),
                "mean_train_r2": cv_results["train_r2"].mean(),
                "mean_val_r2": cv_results["test_r2"].mean(),
            }
        )
    return pd.DataFrame(results)


In [ ]:
cv_results = tree_cross_validation(tree_params, X_train, y_train)
cv_results = cv_results.sort_values(by="mean_val_r2", ascending=False)
display(cv_results.head(7))

,model__max_depth,model__min_samples_leaf,mean_train_rmse,mean_val_rmse,mean_train_r2,mean_val_r2
12,36,4,0.015678,0.033251,0.979978,0.909769
16,22,4,0.015787,0.033265,0.979696,0.909686
14,24,6,0.020148,0.033298,0.966940,0.909555
3,43,7,0.021781,0.034380,0.961357,0.903617
13,40,2,0.008054,0.034621,0.994717,0.902299
28,25,8,0.023279,0.035400,0.955866,0.897658
5,21,8,0.023339,0.035413,0.955637,0.897595


Cross-validation showed that max_depth=36 and min_samples_leaf=4 provided the best generalization, achieving a mean validation RMSE of 0.0333 and R² of 0.9098. This outperformed the previously selected 40/2 configuration and reduced the train–validation performance gap, indicating better generalization with less overfitting.

In [ ]:
def final_tree_pipeline()->Pipeline:
    transformer = mp.feature_transformer()
    tree_pipeline = Pipeline(
        steps=[("preprocessore", transformer),
                ("model", DecisionTreeRegressor(random_state=42, min_samples_leaf=4, max_depth=36))
            ])
    return tree_pipeline

tree_pipeline = final_tree_pipeline()

In [ ]:
def final_tree_model(X_train, y_train, X_test ) -> tuple[Pipeline, np.ndarray]:
    final_tree_model = tree_pipeline.fit(X_train, y_train)
    y_final_tree_pred = final_tree_model.predict(X_test)
    return final_tree_model, y_final_tree_pred

final_tree_model, y_final_tree_pred = final_tree_model(X_train, y_train, X_test )
display(final_tree_model)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessore', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Elevation','Latitude','Longitude',...,'Pollution_level','Plot_size', 'Crop_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subs

## Final Model Evaluation

In [ ]:
def final_evaluation(y_test, y_final_tree_pred) ->tuple[float, float]:
    final_tree_rmse =root_mean_squared_error(y_test, y_final_tree_pred)
    final_tree_r2 = r2_score(y_test, y_final_tree_pred)
    return final_tree_rmse, final_tree_r2

final_tree_rmse, final_tree_r2 = final_evaluation(y_test, y_final_tree_pred)

print(f"RMSE: {final_tree_rmse:.4f}")
print(f"r2 score: {final_tree_r2:.4f}")

RMSE: 0.0322
r2 score: 0.9223


## Model Performance Comparison

| Model | RMSE | R² |
|---|---:|---:|
| OLS / Linear Regression | 0.0678 | 0.6553 |
| RidgeCV | 0.0678 | 0.6552 |
| LassoCV | 0.0699 | 0.6333 |
| Polynomial Regression | 0.0589 | 0.7396 |
| Decision Tree | **0.0322** | **0.9223** |

### Insight

The model comparison shows that nonlinear approaches substantially outperform the linear and regularized models. Polynomial Regression and Decision Tree are able to capture nonlinear relationships between the agricultural features and `Standard_yield` that OLS, Ridge, and Lasso cannot capture effectively.

The **Decision Tree currently achieves the strongest performance**, with a test RMSE of **0.0322** and an R² of **0.9223**. This suggests that nonlinear relationships are important in predicting `Standard_yield` and that tree-based models may be better suited to this dataset than the linear approaches evaluated so far.